# Top pan-cancer high-disruption sample-gene pairs by modality

This notebook rebuilds modality-then-gene normalization for each disruption modality. For every modality, it selects pairs assigned to **High disruption, no hotspot** and ranks the top 10 by `zscore_modality_gene`. This ranking uses only each modality's gene-level normalized disruption value; it does not apply TMB adjustment.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

PROJECT_ROOT = next(path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / 'data').is_dir() and (path / 'notebooks').is_dir())
PANCANCER_SCORE_PATH = PROJECT_ROOT / 'data' / 'alphagenome_scores' / 'patient_tissue_tracks_local' / 'patient_tissue_track_scores.csv'
ALPHAMISSENSE_SCORE_PATH = PROJECT_ROOT / 'data' / 'alphamissense_combined_gene_level_scores.tsv'
SURVIVAL_PATH = PROJECT_ROOT / 'data' / 'TCGA' / 'survival' / 'TCGA_survival_outcome.csv'
TMB_DIR = PROJECT_ROOT / 'data' / 'TCGA' / 'tcga_patient_variants_by_cancer'
HOTSPOT_VARIANT_DIR = PROJECT_ROOT / 'data' / 'TCGA' / 'tcga_driver_gene_hotspot_variants_by_cancer'
PANCANCER_DRIVER_GENE_PATH = PROJECT_ROOT / 'data' / 'driver_genes_coords' / 'Pancancer_1pc.tsv'
PANCANCER_COMPLETION_LEDGER_PATH = PANCANCER_SCORE_PATH.with_name('patient_tissue_track_scores.completed_patient_genes.tsv')
PANCANCER_TRACK_MAPPING_PATH = PROJECT_ROOT / 'metadata' / 'cancer_tissue_alphagenome_tracks.tsv'
ATAC_SAMPLE_DIR = PROJECT_ROOT / 'metadata' / 'tcga_atac_seq_gdc'
OUTPUT_DIR = PROJECT_ROOT / 'results' / 'tcga_pancancer_hierarchically_normalized_tmb_adjusted_hazard_ratios'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SELECTED_CANCER_TYPES = None
SELECTED_GENES = None
SELECTED_MODALITIES = ['RNA_SEQ', 'ATAC', 'CHIP_TF', 'SPLICE_SITE_USAGE']
SCORE_AGGREGATION_TYPE = 'L2_DIFF'
PANCANCER_SCORE_CHUNKSIZE = 500_000
RANKING_SCORE = 'zscore_modality_gene'
ZERO_TOLERANCE = 0.0
NORMALIZATION_LEVELS = ['modality', 'gene']
PRIMARY_ALPHAGENOME_MODALITIES = ['ATAC', 'RNA_SEQ', 'CHIP_TF', 'SPLICE_SITE_USAGE']
TOP_N = 10

def normalize_modality(value):
    return str(value).strip().upper().replace('-', '_')

def normalize_selection(value):
    if value is None:
        return None
    return [str(item).strip() for item in ([value] if isinstance(value, str) else value) if str(item).strip()]

def zscore(values):
    values = pd.to_numeric(values, errors='coerce')
    std = values.std(ddof=0)
    return (values - values.mean()) / std if pd.notna(std) and std > 0 else values * 0.0


/Users/an36943/miniforge3/envs/cancer-model/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_survival():
    survival = pd.read_csv(SURVIVAL_PATH, dtype={'bcr_patient_barcode': str}).rename(columns={'bcr_patient_barcode': 'patient_id', 'type': 'cancer', 'age_at_initial_pathologic_diagnosis': 'age'})
    for column in ['OS.time', 'OS', 'age']:
        survival[column] = pd.to_numeric(survival[column], errors='coerce')
    return survival.dropna(subset=['patient_id', 'cancer', 'OS.time', 'OS']).query('`OS.time` > 0')[['patient_id', 'cancer', 'OS.time', 'OS', 'age']]

def load_tmb(cancer):
    try:
        tmb = pd.read_csv(TMB_DIR / f'{cancer}_tmb.tsv', sep='\t', dtype={'bcr_patient_barcode': str}).rename(columns={'bcr_patient_barcode': 'patient_id'})
    except pd.errors.EmptyDataError:
        return pd.DataFrame(columns=['patient_id', 'tmb', 'total_mutations'])
    tmb['total_mutations'] = pd.to_numeric(tmb['total_mutations'], errors='coerce')
    tmb = tmb.dropna(subset=['patient_id', 'total_mutations'])
    tmb['tmb'] = np.log1p(tmb['total_mutations'])
    return tmb[['patient_id', 'tmb', 'total_mutations']].drop_duplicates('patient_id')

def load_hotspots(cancer, genes):
    variants = pd.read_csv(HOTSPOT_VARIANT_DIR / f'{cancer}_variants.tsv', sep='\t', dtype=str, usecols=['bcr_patient_barcode', 'gene_name', 'is_hotspot']).rename(columns={'bcr_patient_barcode': 'patient_id', 'gene_name': 'gene'})
    variants = variants[variants['gene'].isin(genes)].copy()
    variants['has_hotspot'] = variants['is_hotspot'].str.lower().eq('true')
    return variants.groupby(['patient_id', 'gene'], as_index=False)['has_hotspot'].any()

def load_available_modalities(cancers):
    tracks = pd.read_csv(PANCANCER_TRACK_MAPPING_PATH, sep='\t', dtype=str)
    tracks = tracks[tracks['cancer_type'].isin(cancers)].copy()
    tracks['modality'] = tracks['output_type'].str.replace('OutputType.', '', regex=False).map(normalize_modality)
    selected = {normalize_modality(x) for x in normalize_selection(SELECTED_MODALITIES)}
    return tracks[tracks['modality'].isin(selected)][['cancer_type', 'modality']].drop_duplicates().rename(columns={'cancer_type': 'cancer'})

def load_atac_patients():
    sheets = [pd.read_csv(path, sep='\t', dtype=str) for path in ATAC_SAMPLE_DIR.glob('gdc_sample_sheet*.tsv')]
    atac = pd.concat(sheets, ignore_index=True)
    atac = atac.loc[atac['Data Type'].eq('Aligned Reads') & atac['Tissue Type'].eq('Tumor') & atac['Tumor Descriptor'].eq('Primary'), ['Case ID', 'Project ID']]
    return atac.assign(patient_id=atac['Case ID'], cancer=atac['Project ID'].str.removeprefix('TCGA-'))[['patient_id', 'cancer']].drop_duplicates()

def load_alphagenome_scores(patient_ids_by_cancer, genes):
    patients = set().union(*patient_ids_by_cancer.values())
    cancers = set(patient_ids_by_cancer)
    modalities = {normalize_modality(x) for x in normalize_selection(SELECTED_MODALITIES)}
    usecols = ['patient_id', 'cancer_type', 'gene', 'gene_strand', 'output_type', 'scorer', 'aggregation_type', 'score', 'track_strand', 'n_window_mutations', 'mutations']
    frames = []
    for chunk in tqdm(pd.read_csv(PANCANCER_SCORE_PATH, usecols=usecols, dtype={'patient_id': str}, chunksize=PANCANCER_SCORE_CHUNKSIZE), desc='Loading AlphaGenome scores'):
        chunk['modality'] = chunk['output_type'].map(normalize_modality)
        chunk = chunk[chunk['cancer_type'].isin(cancers) & chunk['patient_id'].isin(patients) & chunk['gene'].isin(genes) & chunk['modality'].isin(modalities) & chunk['scorer'].eq('PATIENT_SINGLE_TRACK_SCORER') & chunk['aggregation_type'].eq(SCORE_AGGREGATION_TYPE)].copy()
        strand_match = chunk['track_strand'].eq(chunk['gene_strand'])
        has_match = strand_match.groupby([chunk[column] for column in ['patient_id', 'cancer_type', 'gene', 'modality']]).transform('any')
        chunk = chunk[(has_match & strand_match) | (~has_match & chunk['track_strand'].eq('.'))]
        frames.append(chunk[['patient_id', 'cancer_type', 'gene', 'modality', 'score', 'n_window_mutations', 'mutations']])
    scores = pd.concat(frames, ignore_index=True).rename(columns={'cancer_type': 'cancer'})
    scores['score'] = pd.to_numeric(scores['score'], errors='coerce')
    return scores.dropna(subset=['score']).groupby(['patient_id', 'cancer', 'gene', 'modality'], as_index=False).agg(gene_disruption=('score', 'median'), n_window_mutations=('n_window_mutations', 'max'), window_mutations=('mutations', 'first'))

def load_alphamissense_scores(patient_ids_by_cancer, genes):
    scores = pd.read_csv(ALPHAMISSENSE_SCORE_PATH, sep='\t', dtype={'bcr_patient_barcode': str}, usecols=['bcr_patient_barcode', 'cancer_type', 'gene', 'combined_pathogenicity_burden']).rename(columns={'bcr_patient_barcode': 'patient_id', 'cancer_type': 'cancer', 'combined_pathogenicity_burden': 'gene_disruption'})
    patients = set().union(*patient_ids_by_cancer.values())
    scores = scores[scores['cancer'].isin(patient_ids_by_cancer) & scores['patient_id'].isin(patients) & scores['gene'].isin(genes)].copy()
    scores['gene_disruption'] = pd.to_numeric(scores['gene_disruption'], errors='coerce')
    scores['modality'] = 'ALPHAMISSENSE'
    return scores.dropna(subset=['gene_disruption']).groupby(['patient_id', 'cancer', 'gene', 'modality'], as_index=False)['gene_disruption'].sum()


In [3]:
def build_analysis_table():
    survival = load_survival()
    available_cancers = sorted(path.name.removesuffix('_tmb.tsv') for path in TMB_DIR.glob('*_tmb.tsv'))
    cancers = available_cancers if SELECTED_CANCER_TYPES is None else normalize_selection(SELECTED_CANCER_TYPES)
    genes = pd.read_csv(PANCANCER_DRIVER_GENE_PATH, sep='\t', dtype=str, usecols=['gene'])['gene'].dropna().drop_duplicates().tolist()
    if SELECTED_GENES is not None:
        genes = [gene for gene in normalize_selection(SELECTED_GENES) if gene in genes]
    tmb_by_cancer = {cancer: load_tmb(cancer) for cancer in cancers}
    patient_ids_by_cancer, pairs, hotspots = {}, [], []
    for cancer in cancers:
        patients = sorted(set(survival.loc[survival['cancer'].eq(cancer), 'patient_id']) & set(tmb_by_cancer[cancer]['patient_id']))
        if patients:
            patient_ids_by_cancer[cancer] = set(patients)
            pairs.append(pd.MultiIndex.from_product([patients, genes], names=['patient_id', 'gene']).to_frame(index=False).assign(cancer=cancer))
            hotspots.append(load_hotspots(cancer, genes))
    pair_df = pd.concat(pairs, ignore_index=True)
    completed = pd.read_csv(PANCANCER_COMPLETION_LEDGER_PATH, sep='\t', dtype=str)
    completed = completed[completed['patient_id'].isin(pair_df['patient_id']) & completed['gene'].isin(genes)][['patient_id', 'gene']].drop_duplicates()
    available = load_available_modalities(patient_ids_by_cancer)
    scores = load_alphagenome_scores(patient_ids_by_cancer, genes).merge(available, on=['cancer', 'modality'], how='inner')
    zeros = pair_df.merge(completed, on=['patient_id', 'gene']).merge(available, on='cancer').assign(gene_disruption=0.0)
    df = pd.concat([scores, zeros], ignore_index=True).drop_duplicates(['patient_id', 'cancer', 'gene', 'modality'], keep='first')
    atac_patients = load_atac_patients()
    df = df.merge(atac_patients.assign(has_atac=True), on=['patient_id', 'cancer'], how='left')
    df['has_atac'] = df['has_atac'].fillna(False)
    tmb = pd.concat([frame.assign(cancer=cancer) for cancer, frame in tmb_by_cancer.items()], ignore_index=True)
    df = df.merge(pd.concat(hotspots, ignore_index=True), on=['patient_id', 'gene'], how='left').merge(survival, on=['patient_id', 'cancer']).merge(tmb, on=['patient_id', 'cancer'])
    return df.assign(has_hotspot=df['has_hotspot'].fillna(False).astype(bool))

def add_normalized_scores(df):
    df = df.copy()
    score_column = 'gene_disruption'
    levels = []
    for level in NORMALIZATION_LEVELS:
        levels.append(level)
        score_column = 'zscore_' + '_'.join(levels)
        df[score_column] = df.groupby(levels)['gene_disruption' if len(levels) == 1 else previous_column].transform(zscore)
        previous_column = score_column
    return df

def assign_disruption_groups(df):
    rows = []
    for _, group in df.groupby(['gene', 'modality'], sort=True):
        group = group.dropna(subset=[RANKING_SCORE]).copy()
        hotspot = group['has_hotspot']
        nonzero = ~hotspot & group['gene_disruption'].gt(ZERO_TOLERANCE)
        group['disruption_group'] = '0 disruption'
        group.loc[hotspot, 'disruption_group'] = 'Hotspot mutation'
        median = group.loc[nonzero, RANKING_SCORE].median()
        group.loc[nonzero & group[RANKING_SCORE].le(median), 'disruption_group'] = 'Low disruption, no hotspot'
        group.loc[nonzero & group[RANKING_SCORE].gt(median), 'disruption_group'] = 'High disruption, no hotspot'
        rows.append(group)
    return pd.concat(rows, ignore_index=True)


In [4]:
raw_df = build_analysis_table()
normalized_df = add_normalized_scores(raw_df)
modeled_modalities = sorted(set(normalized_df['modality']) - {'OVERALL_ALPHAGENOME'})
grouped_df = assign_disruption_groups(normalized_df[normalized_df['modality'].isin(modeled_modalities)])
candidate_df = grouped_df.loc[~grouped_df['modality'].eq('ATAC') | grouped_df['has_atac']]

top_10_high_disruption_by_modality = (
    candidate_df.loc[candidate_df['disruption_group'].eq('High disruption, no hotspot')]
    .sort_values(['modality', RANKING_SCORE], ascending=[True, False])
    .groupby('modality', group_keys=False)
    .head(TOP_N)
    .loc[:, ['patient_id', 'cancer', 'gene', 'modality', 'gene_disruption', 'n_window_mutations', 'tmb', RANKING_SCORE]]
    .rename(columns={RANKING_SCORE: 'normalized_gene_disruption'})
    .assign(modality_rank=lambda df: df.groupby('modality').cumcount() + 1)
    .loc[:, ['modality', 'modality_rank', 'patient_id', 'cancer', 'gene', 'gene_disruption', 'n_window_mutations', 'tmb', 'normalized_gene_disruption']]
    .reset_index(drop=True)
)

output_path = OUTPUT_DIR / 'top_10_high_disruption_no_hotspot_sample_gene_pairs_by_modality.tsv'
top_10_high_disruption_by_modality.to_csv(output_path, sep='\t', index=False)
print(f'Saved {len(top_10_high_disruption_by_modality)} pairs to {output_path}')
top_10_high_disruption_by_modality


Loading AlphaGenome scores: 34it [00:59,  1.76s/it]
/var/folders/kb/00sgh9j520x459vtmckmbyjw0000gp/T/ipykernel_73106/2511142598.py:25: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['has_atac'] = df['has_atac'].fillna(False)
/var/folders/kb/00sgh9j520x459vtmckmbyjw0000gp/T/ipykernel_73106/2511142598.py:26: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  tmb = pd.concat([frame.assign(cancer=cancer) for cancer, frame in tmb_by_cancer.items()], ignore_index=True)
/var/folders/kb/00sgh9j520x459vtmckmbyjw0000gp/T/ipykernel_73106/

Saved 40 pairs to /Users/an36943/PhD/Projects/OncoGenie/results/tcga_pancancer_hierarchically_normalized_tmb_adjusted_hazard_ratios/top_10_high_disruption_no_hotspot_sample_gene_pairs_by_modality.tsv


,modality,modality_rank,patient_id,cancer,gene,gene_disruption,n_window_mutations,tmb,normalized_gene_disruption
0,ATAC,1,TCGA-BL-A13J,BLCA,FAT1,30.598700,494.0,13.694723,10.269559
1,ATAC,2,TCGA-AA-A00E,COAD,SETD2,15.698097,524.0,13.860181,9.888245
2,ATAC,3,TCGA-BJ-A3PU,THCA,CSMD3,8.722718,2066.0,15.215655,8.103974
3,ATAC,4,TCGA-AD-6889,COAD,SMAD4,18.810926,410.0,13.920269,7.936187
4,ATAC,5,TCGA-NH-A6GC,COAD,SMAD4,18.680202,19.0,10.977602,7.873826
5,ATAC,6,TCGA-BJ-A3PU,THCA,FAT1,22.584085,1749.0,15.215655,7.275393
6,ATAC,7,TCGA-XC-AA0X,LUSC,CTNNB1,16.015008,22.0,11.059519,6.886478
7,ATAC,8,TCGA-BR-A4IY,STAD,CDKN2A,7.301661,13.0,10.634990,6.868890
8,ATAC,9,TCGA-A6-A56B,COAD,KMT2D,12.521606,43.0,11.969388,6.760618
9,ATAC,10,TCGA-BJ-A3PU,THCA,KMT2C,17.603655,1800.0,15.215655,5.585911
